In [1]:
# Step 1: Install required libraries
# datasets==3.6.0 is used to avoid the conll2003 loading error

!pip install -q datasets==3.6.0 transformers accelerate seqeval evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00


In [3]:
# Step 2: Import required libraries

import numpy as np
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import classification_report

In [4]:
# Step 3: Load dataset
# This dataset contains tokens, POS tags, chunk tags, and NER tags

dataset = load_dataset("conll2003")

print(dataset)
print("\nFirst training example:")
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

conll2003.py: 0.00B [00:00, ?B/s]

The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

First training example:
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [5]:
# Step 4: Extract label names for POS tagging and Chunking

pos_label_list = dataset["train"].features["pos_tags"].feature.names
chunk_label_list = dataset["train"].features["chunk_tags"].feature.names

print("POS Labels:")
print(pos_label_list)

print("\nChunk Labels:")
print(chunk_label_list)

POS Labels:
['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']

Chunk Labels:
['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [6]:
# Step 5: Load tokenizer
# DistilBERT is lighter and faster than full BERT

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# Step 6: Tokenize and align POS labels with subword tokens

def tokenize_and_align_labels_pos(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens get -100
            if word_idx is None:
                label_ids.append(-100)
            # First token of each word gets the real label
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            # Subword tokens get -100
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [8]:
# Step 7: Apply POS preprocessing

tokenized_pos = dataset.map(tokenize_and_align_labels_pos, batched=True)

print(tokenized_pos)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})


In [9]:
# Step 8: Define seqeval metrics for POS tagging

seqeval = evaluate.load("seqeval")

def compute_metrics_pos(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        pred_labels = []
        true_labs = []

        for p, l in zip(prediction, label):
            if l != -100:
                pred_labels.append(pos_label_list[p])
                true_labs.append(pos_label_list[l])

        true_predictions.append(pred_labels)
        true_labels.append(true_labs)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [10]:
# Step 9: Load DistilBERT model for POS tagging

pos_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(pos_label_list),
    id2label={i: label for i, label in enumerate(pos_label_list)},
    label2id={label: i for i, label in enumerate(pos_label_list)}
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# Step 10: Data collator for token classification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [12]:
# Step 11: Select smaller subsets for faster training in Colab

small_train_pos = tokenized_pos["train"].select(range(2000))
small_val_pos = tokenized_pos["validation"].select(range(500))
small_test_pos = tokenized_pos["test"].select(range(200))

In [13]:
# Step 12: Training arguments for POS tagging

pos_training_args = TrainingArguments(
    output_dir="./pos_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

In [14]:
# Step 13: Create Trainer for POS model

pos_trainer = Trainer(
    model=pos_model,
    args=pos_training_args,
    train_dataset=small_train_pos,
    eval_dataset=small_val_pos,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_pos
)

In [15]:
# Step 14: Train POS tagging model

pos_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.824996,0.681195,0.795286,0.788911,0.792086,0.858421


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=1.5658365783691406, metrics={'train_runtime': 21.3982, 'train_samples_per_second': 93.466, 'train_steps_per_second': 11.683, 'total_flos': 20688589861728.0, 'train_loss': 1.5658365783691406, 'epoch': 1.0})

In [16]:
# Step 15: Evaluate POS tagging model

pos_results = pos_trainer.evaluate()
print("POS Tagging Results:")
print(pos_results)

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

POS Tagging Results:
{'eval_loss': 0.6811954975128174, 'eval_precision': 0.7952861952861953, 'eval_recall': 0.7889111556446226, 'eval_f1': 0.7920858484238766, 'eval_accuracy': 0.858421052631579, 'eval_runtime': 1.2724, 'eval_samples_per_second': 392.971, 'eval_steps_per_second': 49.514, 'epoch': 1.0}


In [17]:
# Step 16: Tokenize and align Chunk labels with subword tokens

def tokenize_and_align_labels_chunk(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [18]:
# Step 17: Apply Chunk preprocessing

tokenized_chunk = dataset.map(tokenize_and_align_labels_chunk, batched=True)

print(tokenized_chunk)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})


In [19]:
# Step 18: Define seqeval metrics for Chunking

def compute_metrics_chunk(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        pred_labels = []
        true_labs = []

        for p, l in zip(prediction, label):
            if l != -100:
                pred_labels.append(chunk_label_list[p])
                true_labs.append(chunk_label_list[l])

        true_predictions.append(pred_labels)
        true_labels.append(true_labs)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [20]:
# Step 19: Load DistilBERT model for Chunking

chunk_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(chunk_label_list),
    id2label={i: label for i, label in enumerate(chunk_label_list)},
    label2id={label: i for i, label in enumerate(chunk_label_list)}
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
# Step 20: Select smaller subsets for faster chunking training

small_train_chunk = tokenized_chunk["train"].select(range(2000))
small_val_chunk = tokenized_chunk["validation"].select(range(500))
small_test_chunk = tokenized_chunk["test"].select(range(200))

In [22]:
# Step 21: Training arguments for Chunking

chunk_training_args = TrainingArguments(
    output_dir="./chunk_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

In [23]:
# Step 22: Create Trainer for Chunking model

chunk_trainer = Trainer(
    model=chunk_model,
    args=chunk_training_args,
    train_dataset=small_train_chunk,
    eval_dataset=small_val_chunk,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_chunk
)

In [24]:
# Step 23: Train Chunking model

chunk_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.422798,0.397158,0.824869,0.814528,0.819666,0.905965


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.817051513671875, metrics={'train_runtime': 19.6768, 'train_samples_per_second': 101.643, 'train_steps_per_second': 12.705, 'total_flos': 20679619359840.0, 'train_loss': 0.817051513671875, 'epoch': 1.0})

In [25]:
# Step 24: Evaluate Chunking model

chunk_results = chunk_trainer.evaluate()
print("Chunking Results:")
print(chunk_results)

Chunking Results:
{'eval_loss': 0.39715757966041565, 'eval_precision': 0.8248693054518297, 'eval_recall': 0.81452802359882, 'eval_f1': 0.8196660482374768, 'eval_accuracy': 0.9059649122807018, 'eval_runtime': 1.1068, 'eval_samples_per_second': 451.737, 'eval_steps_per_second': 56.919, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [27]:
# Step 25: Inference on custom sentence

import torch

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move models to the same device
pos_model.to(device)
chunk_model.to(device)

sentence = "John works at Google in California"
words = sentence.split()

# Tokenize input sentence
inputs = tokenizer(
    words,
    return_tensors="pt",
    is_split_into_words=True,
    truncation=True
)

# Move inputs to the same device as the model
inputs = {key: value.to(device) for key, value in inputs.items()}

# POS prediction
with torch.no_grad():
    pos_outputs = pos_model(**inputs)
pos_preds = torch.argmax(pos_outputs.logits, dim=2)[0].cpu().tolist()

# Chunk prediction
with torch.no_grad():
    chunk_outputs = chunk_model(**inputs)
chunk_preds = torch.argmax(chunk_outputs.logits, dim=2)[0].cpu().tolist()

# Convert tokens back to readable form
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

print("Tokens:")
print(tokens)

print("\nPOS Predictions:")
for token, pred in zip(tokens, pos_preds):
    print(token, "->", pos_label_list[pred])

print("\nChunk Predictions:")
for token, pred in zip(tokens, chunk_preds):
    print(token, "->", chunk_label_list[pred])

Tokens:
['[CLS]', 'john', 'works', 'at', 'google', 'in', 'california', '[SEP]']

POS Predictions:
[CLS] -> NNP
john -> NNP
works -> VBZ
at -> IN
google -> NNP
in -> IN
california -> NNP
[SEP] -> .

Chunk Predictions:
[CLS] -> B-NP
john -> B-NP
works -> B-VP
at -> B-PP
google -> B-NP
in -> B-PP
california -> B-NP
[SEP] -> O


## Comparison: POS Tagging vs Chunking

POS tagging assigns grammatical labels to individual words such as noun, verb, adjective, or pronoun. It focuses on the role of each word in a sentence.

Chunking groups words into meaningful phrases such as noun phrases or verb phrases. It works at phrase level rather than only word level.

POS tagging is generally easier because it deals with grammar-level classification. Chunking is slightly more complex because it identifies phrase boundaries and label transitions such as B-NP and I-NP.

In this project, both tasks were implemented using DistilBERT with token classification. The transformer model handled context-aware predictions for both POS tags and chunk tags.

## Report: Observations and Challenges

In this project, DistilBERT was fine-tuned for two token classification tasks: POS tagging and chunking.

One important challenge was label alignment. Since the tokenizer can split one word into multiple subwords, labels had to be aligned carefully. Special tokens and extra subword pieces were assigned -100 so they would be ignored during loss calculation.

Another observation is that POS tagging focuses on word-level grammar, while chunking focuses on phrase-level structure. Chunking is more difficult because it requires the model to learn boundaries between phrases.

Overall, this task helped in understanding token classification, subword tokenization, label alignment, and transformer-based sequence labeling.